# **AI-Driven Network Intrusion Detection Pipeline**

**Author:** Azeeza Agrippina Lesmana

**Description:** End-to-end pipeline for processing 100k+ network logs,
             applying Isolation Forest for anomaly detection, and
             formatting for Elasticsearch/Kibana ingestion.

In [16]:
import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import IsolationForest
from datetime import datetime, timedelta

# --- STEP 1: AUTOMATIC DATASET DOWNLOAD ---
print("--- Downloading dataset from Kaggle ---")
path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")
csv_file = os.path.join(path, 'Wednesday-workingHours.pcap_ISCX.csv')

# Load 100k records for analysis [cite: 5]
df = pd.read_csv(csv_file, low_memory=False, nrows=100000)

# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# --- STEP 2: MACHINE LEARNING (ISOLATION FOREST) ---
# Features aligned with Telecommunication Engineering standards [cite: 5]
features = [
    'flow_duration', 'total_fwd_packets', 'total_backward_packets',
    'fwd_packet_length_max', 'bwd_packet_length_max'
]

for col in features:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

# Unsupervised Anomaly Detection
model = IsolationForest(n_estimators=100, contamination=0.03, random_state=42)
model.fit(df[features])

df['anomaly_score'] = model.decision_function(df[features])
df['is_anomaly'] = model.predict(df[features])
df['is_anomaly'] = df['is_anomaly'].apply(lambda x: 'true' if x == -1 else 'false')

# --- STEP 3: TIMESTAMP & FORMATTING FOR ELASTICSTACK ---
start_time = datetime.now()
df['timestamp'] = [
    (start_time + timedelta(seconds=i)).strftime('%Y-%m-%dT%H:%M:%SZ')
    for i in range(len(df))
]

if 'label' in df.columns:
    df.rename(columns={'label': 'attack_type'}, inplace=True)

final_cols = ['timestamp', 'destination_port', 'attack_type', 'anomaly_score', 'is_anomaly']
final_df = df[[col for col in final_cols if col in df.columns]].copy()

# --- STEP 4: EXPORT & DOWNLOAD ---
output_filename = 'elastic_ready_data.csv'
final_df.to_csv(output_filename, index=False)

print(f"--- SUCCESS! File '{output_filename}' created ---")

# THIS LINE TRIGGERS THE DOWNLOAD IN GOOGLE COLAB
try:
    from google.colab import files
    files.download(output_filename)
except ImportError:
    print(f"Not in Colab. File saved locally as: {os.path.abspath(output_filename)}")

--- Downloading dataset from Kaggle ---
Using Colab cache for faster access to the 'network-intrusion-dataset' dataset.
--- SUCCESS! File 'elastic_ready_data.csv' created ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>